# Практическое задание №3
## Визуализация графов. Интерактивная визуализация в Plotly и Pyvis.

---

**Датасет:** FinFraud_Labelled.csv - Журналы транзакций системы мобильных денежных переводов (МДП)

**Цель:** Исследовать графовые структуры транзакций, выявить паттерны мошеннических схем и визуализировать их с помощью Plotly и Pyvis.

**Выполнил:** Кочуров Александр Дмитриевич 2384


## 1. Загрузка и исследование данных

In [78]:
import pandas as pd
import numpy as np
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pyvis.network import Network
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [79]:
columns = [
    'label',          # 0 - метка (F-... = мошенничество)
    'sender_id',      # 1 - ID отправителя
    'receiver_id',    # 2 - ID получателя
    'sender_acc',     # 3 - счёт отправителя
    'receiver_acc',   # 4 - счёт получателя
    'amount',         # 5 - сумма
    'tx_type',        # 6 - тип транзакции
    'status',         # 7 - статус (SU/FA)
    'sender_bal_before',   # 8
    'sender_bal_after',    # 9
    'receiver_bal_before', # 10
    'receiver_bal_after',  # 11
    'flag1', 'flag2', 'flag3', 'flag4',  # 12-15 не используются
    'ts_sender',      # 16 - метка времени (отправитель)
    'ts_receiver',    # 17 - метка времени (получатель)
    'sender_acc2',    # 18
    'srv1', 'srv2',   # 19-20 служебные
    'tx_id',          # 21 - ID транзакции
    'ts_tx',          # 22 - время транзакции
    'sender_type',    # 23 - тип отправителя
    'receiver_type'   # 24 - тип получателя
]

df = pd.read_csv(
    'FinFraud_Labelled.csv',
    sep='|',
    header=None,
    names=columns,
    on_bad_lines='skip'
)

# Убираем пустые строки
df = df.dropna(subset=['label', 'sender_id', 'receiver_id'])
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df['is_fraud'] = df['label'].str.startswith('F')

print(f'Загружено строк: {len(df)}')
print(f'Из них мошеннических: {df["is_fraud"].sum()}')
print(f'Легитимных: {(~df["is_fraud"]).sum()}')

Загружено строк: 54848
Из них мошеннических: 1678
Легитимных: 53170


In [80]:
# Распределение меток транзакций
label_counts = df['label'].value_counts()
print('Метки транзакций:')
print(label_counts)

# Типы транзакций
print('\nТипы транзакций:')
print(df['tx_type'].value_counts())

Метки транзакций:
label
N_Reg_RC       28312
N-RegDep       12867
N-RegC2C        7484
N_RegWith       4064
F_bot            721
F-Mule-With      717
N_Reg_Merch      443
F_SevWith        240
Name: count, dtype: int64

Типы транзакций:
tx_type
ArRC        28312
Dt          12867
Ind          8205
Wl           5021
Merchant      443
Name: count, dtype: int64


In [81]:
# Интерактивная диаграмма: распределение меток транзакций
label_map = {
    'N-RegC2C':    'Легитимный перевод (C2C)',
    'N-RegDep':    'Легитимное пополнение',
    'N_RegWith':   'Легитимное снятие',
    'N_Reg_RC':    'Легитимная арRC',
    'N_Reg_Merch': 'Легитимный мерчант',
    'F_bot':       'X Ботнет (заражение)',
    'F-Mule-With': 'X Мул-вывод средств',
    'F_SevWith':   'X Кража телефона (многократное снятие)',
}
colors_pie = [
    '#3B82F6','#60A5FA','#93C5FD','#BFDBFE','#DBEAFE',
    '#EF4444','#DC2626','#B91C1C'
]
labels_display = [label_map.get(l, l) for l in label_counts.index]

fig = go.Figure(go.Pie(
    labels=labels_display,
    values=label_counts.values,
    marker_colors=colors_pie,
    hole=0.4,
    textinfo='label+percent',
    hovertemplate='<b>%{label}</b><br>Кол-во: %{value}<br>Доля: %{percent}<extra></extra>'
))
fig.update_layout(
    title='Распределение типов транзакций в системе МДП',
    title_font_size=18,
    legend=dict(orientation='v', x=1.02),
    margin=dict(t=60, b=20)
)
fig.show()

## 2. Построение графа транзакций

In [82]:
# Строим граф по всем транзакциям
G_full = nx.DiGraph()

for _, row in df.iterrows():
    src = row['sender_id']
    dst = row['receiver_id']
    if pd.isna(src) or pd.isna(dst):
        continue
    if G_full.has_edge(src, dst):
        G_full[src][dst]['weight'] += 1
        G_full[src][dst]['amount'] += row['amount'] if not pd.isna(row['amount']) else 0
        if row['is_fraud']:
            G_full[src][dst]['fraud_count'] = G_full[src][dst].get('fraud_count', 0) + 1
    else:
        G_full.add_edge(
            src, dst,
            weight=1,
            amount=row['amount'] if not pd.isna(row['amount']) else 0,
            tx_type=row['tx_type'],
            label=row['label'],
            fraud_count=1 if row['is_fraud'] else 0
        )

# Добавляем атрибуты узлов
sender_types = df.groupby('sender_id')['sender_type'].first().to_dict()
receiver_types = df.groupby('receiver_id')['receiver_type'].first().to_dict()
node_types = {**receiver_types, **sender_types}  # sender перекроет receiver при совпадении

nx.set_node_attributes(G_full, node_types, 'node_type')

print(f'Граф: {G_full.number_of_nodes()} узлов, {G_full.number_of_edges()} рёбер')

Граф: 2009 узлов, 7476 рёбер


In [83]:
in_deg = dict(G_full.in_degree())
out_deg = dict(G_full.out_degree())

top_in = sorted(in_deg.items(), key=lambda x: x[1], reverse=True)[:15]
top_out = sorted(out_deg.items(), key=lambda x: x[1], reverse=True)[:15]

print('TOP-15 по входящим связям (получатели многих переводов):')
for node, deg in top_in:
    ntype = node_types.get(node, '?')
    print(f'  {node} [{ntype}]: {deg}')

print('\nTOP-15 по исходящим связям (отправители в много мест):')
for node, deg in top_out:
    ntype = node_types.get(node, '?')
    print(f'  {node} [{ntype}]: {deg}')

TOP-15 по входящим связям (получатели многих переводов):
  operator [operator]: 1550
  PN_Ret4 [RET]: 197
  PN_Ret6 [RET]: 196
  PN_Ret5 [RET]: 189
  PN_Ret2 [RET]: 178
  PN_Ret3 [RET]: 177
  PN_Ret1 [RET]: 162
  PN_EU_0_260 [EU]: 44
  PN_EU_0_955 [EU]: 42
  PN_EU_1_328 [EU]: 40
  PN_EU_0_1045 [EU]: 39
  PN_MER1 [MER]: 15
  PN_MER2 [MER]: 15
  PN_EU_1_267 [EU]: 9
  PN_EU_2_120 [EU]: 9

TOP-15 по исходящим связям (отправители в много мест):
  PN_Ret1 [RET]: 461
  PN_Ret2 [RET]: 459
  PN_Ret6 [RET]: 448
  PN_Ret3 [RET]: 448
  PN_Ret5 [RET]: 441
  PN_Ret4 [RET]: 438
  PN_EU_3_10 [EU]: 19
  PN_EU_1_278 [EU]: 18
  PN_EU_2_47 [EU]: 17
  PN_EU_2_48 [EU]: 17
  PN_EU_1_167 [EU]: 17
  PN_EU_2_36 [EU]: 16
  PN_EU_2_31 [EU]: 16
  PN_EU_3_17 [EU]: 15
  PN_EU_2_51 [EU]: 15


In [84]:
# Интерактивный Plotly: степени узлов (топ-30)
all_nodes = list(set(list(in_deg.keys()) + list(out_deg.keys())))
node_df = pd.DataFrame({
    'node': all_nodes,
    'in_deg': [in_deg.get(n, 0) for n in all_nodes],
    'out_deg': [out_deg.get(n, 0) for n in all_nodes],
    'node_type': [node_types.get(n, '?') for n in all_nodes]
})
node_df['total_deg'] = node_df['in_deg'] + node_df['out_deg']
node_df = node_df.sort_values('total_deg', ascending=False).head(30)

color_map = {'EU': '#3B82F6', 'RET': '#F59E0B', 'Mer': '#10B981', '?': '#9CA3AF'}
node_df['color'] = node_df['node_type'].map(lambda x: color_map.get(x, '#9CA3AF'))

fig2 = go.Figure()
for ntype, color in color_map.items():
    subset = node_df[node_df['node_type'] == ntype]
    if len(subset) == 0:
        continue
    fig2.add_trace(go.Bar(
        name=ntype,
        x=subset['node'],
        y=subset['in_deg'],
        marker_color=color,
        offsetgroup=ntype,
        hovertemplate='<b>%{x}</b><br>Входящих: %{y}<extra></extra>'
    ))

fig2.update_layout(
    title='Топ-30 узлов по суммарной степени (входящие связи)',
    xaxis_title='Узел', yaxis_title='Входящая степень',
    xaxis_tickangle=-45, barmode='group',
    height=500
)
fig2.show()

## 3. Схема мошенничества №1: Ботнет + Мул (F_bot + F-Mule-With)

### Описание схемы
Мобильный ботнет заражает телефоны обычных пользователей (EU). Вредоносное ПО незаметно для владельца переводит деньги со счёта жертвы на счёт «мула» - специально созданного или скомпрометированного аккаунта. Мул затем быстро снимает деньги через агента-ретейлера (RET).

**Структура:** `Жертва_EU --[F_bot]--> Мул_EU --[F-Mule-With]--> Агент_RET`

In [85]:
# Выделяем мошеннические транзакции ботнета и вывода через мула
df_bot = df[df['label'] == 'F_bot'].copy()
df_mule = df[df['label'] == 'F-Mule-With'].copy()

# Мулы - это получатели ботнет-транзакций, и одновременно отправители при выводе
mule_accounts = set(df_bot['receiver_id']) & set(df_mule['sender_id'])
print(f'Обнаружено мул-аккаунтов: {len(mule_accounts)}')
for m in sorted(mule_accounts):
    bot_count = len(df_bot[df_bot['receiver_id'] == m])
    with_count = len(df_mule[df_mule['sender_id'] == m])
    total_recv = df_bot[df_bot['receiver_id'] == m]['amount'].sum()
    total_sent = df_mule[df_mule['sender_id'] == m]['amount'].sum()
    print(f'  {m}: получил {bot_count} ботнет-переводов ({total_recv:,.0f} mM), вывел {with_count} раз ({total_sent:,.0f} mM)')

Обнаружено мул-аккаунтов: 4
  PN_EU_0_1045: получил 172 ботнет-переводов (1,836,441 mM), вывел 170 раз (1,800,402 mM)
  PN_EU_0_260: получил 167 ботнет-переводов (1,873,295 mM), вывел 167 раз (1,854,562 mM)
  PN_EU_0_955: получил 199 ботнет-переводов (2,057,445 mM), вывел 197 раз (2,018,313 mM)
  PN_EU_1_328: получил 183 ботнет-переводов (1,939,990 mM), вывел 183 раз (1,920,590 mM)


In [86]:
# Строим подграф одной мул-схемы для примера (PN_EU_0_260)
mule_example = 'PN_EU_0_260'

# Берём все транзакции, связанные с этим мулом
df_mule_sub = pd.concat([
    df_bot[df_bot['receiver_id'] == mule_example],
    df_mule[df_mule['sender_id'] == mule_example]
])

G_scheme1 = nx.DiGraph()
for _, row in df_mule_sub.iterrows():
    src = row['sender_id']
    dst = row['receiver_id']
    G_scheme1.add_edge(src, dst,
        amount=row['amount'],
        label=row['label'],
        tx_type=row['tx_type']
    )

# Задаём роли узлов
roles = {}
for n in G_scheme1.nodes():
    if n == mule_example:
        roles[n] = 'mule'
    elif n.startswith('PN_Ret') or n.startswith('RAcc'):
        roles[n] = 'ret'
    else:
        roles[n] = 'victim'

print(f'Узлов в схеме: {G_scheme1.number_of_nodes()}')
print(f'  Жертв: {sum(1 for r in roles.values() if r=="victim")}')
print(f'  Мулов: {sum(1 for r in roles.values() if r=="mule")}')
print(f'  Агентов (RET): {sum(1 for r in roles.values() if r=="ret")}')

Узлов в схеме: 46
  Жертв: 39
  Мулов: 1
  Агентов (RET): 6


In [87]:
# Визуализация схемы 1 через Plotly
import math

# Многоуровневое расположение: жертвы слева, мул в центре, агенты справа
victims = [n for n, r in roles.items() if r == 'victim']
mules = [n for n, r in roles.items() if r == 'mule']
rets = [n for n, r in roles.items() if r == 'ret']

pos = {}
# Жертвы - левая колонка
for i, v in enumerate(sorted(victims)):
    pos[v] = (-2, i - len(victims)/2)
# Мул - центр
for m in mules:
    pos[m] = (0, 0)
# Агенты - правая колонка
for i, r in enumerate(sorted(rets)):
    pos[r] = (2, i - len(rets)/2)

# Цвета
role_color = {'victim': '#3B82F6', 'mule': '#EF4444', 'ret': '#F59E0B'}
role_name = {'victim': 'Жертва (EU)', 'mule': 'Мул (EU)', 'ret': 'Агент (RET)'}
role_size = {'victim': 12, 'mule': 25, 'ret': 18}

node_x = [pos[n][0] for n in G_scheme1.nodes()]
node_y = [pos[n][1] for n in G_scheme1.nodes()]
node_colors = [role_color[roles[n]] for n in G_scheme1.nodes()]
node_sizes = [role_size[roles[n]] for n in G_scheme1.nodes()]
node_labels = list(G_scheme1.nodes())

edge_traces = []
for u, v, data in G_scheme1.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    color = '#EF4444' if data.get('label') == 'F_bot' else '#F59E0B'
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode='lines',
        line=dict(width=1.5, color=color),
        hoverinfo='none',
        showlegend=False
    ))

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    marker=dict(size=node_sizes, color=node_colors, line=dict(width=1.5, color='white')),
    text=[n.replace('PN_EU_', 'EU_').replace('PN_Ret', 'RET') for n in node_labels],
    textposition='top center',
    textfont=dict(size=8),
    hovertemplate='<b>%{text}</b><br>Роль: ' +
        '<br>'.join([f'{n}: {role_name[roles[n]]}' for n in node_labels]) +
        '<extra></extra>',
    showlegend=False
)

# Легенда
legend_traces = [
    go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(size=12, color='#3B82F6'), name='Жертва-EU (телефон заражён)'),
    go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(size=12, color='#EF4444'), name='Мул-аккаунт EU'),
    go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(size=12, color='#F59E0B'), name='Агент-RET (вывод кэша)'),
    go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color='#EF4444', width=2), name='F_bot (ботнет-перевод)'),
    go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color='#F59E0B', width=2), name='F-Mule-With (вывод через мула)'),
]

# Аннотации
annotations = [
    dict(x=-2, y=max(pos[v][1] for v in victims)+1.5, text='<b>Жертвы</b><br>(заражённые EU)',
         showarrow=False, font=dict(size=13, color='#3B82F6'), bgcolor='rgba(59,130,246,0.1)'),
    dict(x=0, y=2.0, text='<b>Мул</b><br>PN_EU_0_260',
         showarrow=False, font=dict(size=13, color='#EF4444'), bgcolor='rgba(239,68,68,0.1)'),
    dict(x=2, y=max(pos[r][1] for r in rets)+1.5, text='<b>Агенты RET</b><br>(обналичивание)',
         showarrow=False, font=dict(size=13, color='#F59E0B'), bgcolor='rgba(245,158,11,0.1)'),
]

fig3 = go.Figure(
    data=edge_traces + [node_trace] + legend_traces,
    layout=go.Layout(
        title=dict(
            text='Схема мошенничества №1: Мобильный ботнет + Мул'
                 '<br><sub>Узел PN_EU_0_260 - центральный мул-аккаунт</sub>',
            font=dict(size=17)
        ),
        showlegend=True,
        legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0.9)', bordercolor='#E5E7EB', borderwidth=1),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        annotations=annotations,
        height=650,
        margin=dict(r=220),
        plot_bgcolor='#F8FAFC'
    )
)
fig3.show()

In [88]:
# Временная шкала мошеннических транзакций для мула PN_EU_0_260
df_mule_time = df_mule_sub.copy()
df_mule_time['ts'] = pd.to_datetime(df_mule_time['ts_sender'], dayfirst=True, errors='coerce')
df_mule_time = df_mule_time.dropna(subset=['ts'])

color_label = {'F_bot': '#EF4444', 'F-Mule-With': '#F59E0B'}

fig4 = go.Figure()
for lbl, grp in df_mule_time.groupby('label'):
    fig4.add_trace(go.Scatter(
        x=grp['ts'], y=grp['amount'],
        mode='markers',
        name=lbl,
        marker=dict(color=color_label.get(lbl, 'grey'), size=8, opacity=0.8),
        hovertemplate='<b>%{x}</b><br>Сумма: %{y:,.0f} mM<extra></extra>'
    ))

fig4.update_layout(
    title='Временная шкала: ботнет-поступления и вывод через мула (PN_EU_0_260)',
    xaxis_title='Дата', yaxis_title='Сумма транзакции (mM)',
    height=400
)
fig4.show()

In [89]:
# Pyvis: интерактивная визуализация ботнет-схемы (сохраняем в HTML)
net1 = Network(height='600px', width='100%', directed=True, bgcolor='#1E293B', font_color='white')
net1.set_options('''
{
  "physics": {
    "barnesHut": {"gravitationalConstant": -8000, "springLength": 200},
    "stabilization": {"iterations": 200}
  },
  "edges": {"arrows": {"to": {"enabled": true, "scaleFactor": 0.8}}}
}
''')

for n in G_scheme1.nodes():
    r = roles[n]
    color = role_color[r]
    title = f"{n}\nРоль: {role_name[r]}"
    size = role_size[r] * 2
    label = n.replace('PN_EU_', 'EU_').replace('PN_Ret', 'RET')
    net1.add_node(n, label=label, color=color, size=size, title=title)

for u, v, data in G_scheme1.edges(data=True):
    lbl = data.get('label', '')
    color = '#EF4444' if lbl == 'F_bot' else '#F59E0B'
    amt = data.get('amount', 0)
    net1.add_edge(u, v, color=color, title=f"{lbl}\n{amt:,.0f} mM", width=2)

net1.save_graph('./outputs/scheme1_botnet_mule.html')

## 4. Схема мошенничества №2: Кража телефона - многократное быстрое снятие (F_SevWith)

### Описание схемы
После кражи смартфона злоумышленник получает доступ к мобильному кошельку жертвы. Цель - как можно быстрее вывести все средства через нескольких агентов-ретейлеров (RET), пока счёт не заблокирован. Ключевые признаки: **один аккаунт EU совершает множество операций снятия Wl в течение нескольких минут** через разных агентов.

**Структура:** `Жертва_EU --[Wl × N]--> Агент_RET_1, RET_2, RET_3...`

In [90]:
df_sev = df[df['label'] == 'F_SevWith'].copy()
df_sev['ts'] = pd.to_datetime(df_sev['ts_sender'], dayfirst=True, errors='coerce')

# Статистика по жертвам
sev_stats = df_sev.groupby('sender_id').agg(
    tx_count=('amount', 'count'),
    total_amount=('amount', 'sum'),
    unique_rets=('receiver_id', 'nunique')
).sort_values('tx_count', ascending=False)

print('Топ жертв кражи телефона (F_SevWith):')
print(sev_stats.head(10).to_string())
print(f'\nВсего жертв: {len(sev_stats)}')
print(f'Среднее кол-во выводов: {sev_stats["tx_count"].mean():.1f}')

Топ жертв кражи телефона (F_SevWith):
              tx_count  total_amount  unique_rets
sender_id                                        
PN_EU_0_1017         4      96844.87            4
PN_EU_0_1044         4      40705.40            3
PN_EU_0_763          4      57348.94            4
PN_EU_0_77           4       9353.13            3
PN_EU_0_81           4      85662.21            3
PN_EU_0_876          4      35379.25            3
PN_EU_0_898          4      91456.61            4
PN_EU_0_90           4      35046.72            3
PN_EU_0_931          4      71177.46            3
PN_EU_0_935          4      28886.03            4

Всего жертв: 60
Среднее кол-во выводов: 4.0


In [91]:
# Строим подграф для нескольких жертв кражи телефона
top_victims_sev = sev_stats.head(6).index.tolist()
df_sev_sub = df_sev[df_sev['sender_id'].isin(top_victims_sev)]

G_scheme2 = nx.DiGraph()
for _, row in df_sev_sub.iterrows():
    src = row['sender_id']
    dst = row['receiver_id']
    if G_scheme2.has_edge(src, dst):
        G_scheme2[src][dst]['weight'] += 1
        G_scheme2[src][dst]['amount'] += row['amount']
    else:
        G_scheme2.add_edge(src, dst, weight=1, amount=row['amount'], label='F_SevWith')

# Роли
roles2 = {}
for n in G_scheme2.nodes():
    if n.startswith('PN_Ret'):
        roles2[n] = 'ret'
    else:
        roles2[n] = 'victim'

print(f'Граф схемы №2: {G_scheme2.number_of_nodes()} узлов, {G_scheme2.number_of_edges()} рёбер')

Граф схемы №2: 12 узлов, 20 рёбер


In [92]:
# Plotly: граф схемы кражи телефона
# Жертвы - верхний полукруг, агенты - нижний ряд
victims2 = sorted([n for n, r in roles2.items() if r == 'victim'])
rets2 = sorted([n for n, r in roles2.items() if r == 'ret'])

pos2 = {}
n_v = len(victims2)
for i, v in enumerate(victims2):
    angle = math.pi * i / max(n_v - 1, 1)
    pos2[v] = (3 * math.cos(angle - math.pi/2), 3 * math.sin(angle - math.pi/2) + 2)
for i, r in enumerate(rets2):
    pos2[r] = (i * 1.2 - (len(rets2)-1)*0.6, -2)

edge_traces2 = []
for u, v, data in G_scheme2.edges(data=True):
    x0, y0 = pos2[u]
    x1, y1 = pos2[v]
    w = min(data.get('weight', 1), 10)
    edge_traces2.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode='lines',
        line=dict(width=w * 0.5 + 0.5, color='rgba(239,68,68,0.6)'),
        hoverinfo='none',
        showlegend=False
    ))

node_x2 = [pos2[n][0] for n in G_scheme2.nodes()]
node_y2 = [pos2[n][1] for n in G_scheme2.nodes()]
node_colors2 = ['#EF4444' if roles2[n] == 'victim' else '#F59E0B' for n in G_scheme2.nodes()]
node_sizes2 = [20 if roles2[n] == 'victim' else 18 for n in G_scheme2.nodes()]
node_text2 = [n.replace('PN_EU_', 'EU_').replace('PN_Ret', 'RET') for n in G_scheme2.nodes()]

# Hover: сколько выводов сделано с аккаунта
hover2 = []
for n in G_scheme2.nodes():
    if roles2[n] == 'victim':
        cnt = sev_stats.loc[n, 'tx_count'] if n in sev_stats.index else 0
        amt = sev_stats.loc[n, 'total_amount'] if n in sev_stats.index else 0
        hover2.append(f'<b>{n}</b><br>Выводов: {cnt}<br>Итого: {amt:,.0f} mM')
    else:
        in_w = sum(d.get('weight',0) for _, _, d in G_scheme2.in_edges(n, data=True))
        hover2.append(f'<b>{n}</b><br>Принял выводов: {in_w}')

node_trace2 = go.Scatter(
    x=node_x2, y=node_y2,
    mode='markers+text',
    marker=dict(size=node_sizes2, color=node_colors2, line=dict(width=1.5, color='white')),
    text=node_text2, textposition='top center', textfont=dict(size=9),
    hovertemplate='%{customdata}<extra></extra>',
    customdata=hover2,
    showlegend=False
)

legend_traces2 = [
    go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(size=12, color='#EF4444'), name='Жертва кражи (EU-аккаунт)'),
    go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(size=12, color='#F59E0B'), name='Агент RET (обналичивание)'),
    go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color='rgba(239,68,68,0.6)', width=3),
        name='Многократное снятие Wl (толщина = кол-во)'),
]

fig5 = go.Figure(
    data=edge_traces2 + [node_trace2] + legend_traces2,
    layout=go.Layout(
        title=dict(
            text='Схема мошенничества №2: Кража телефона (F_SevWith)'
                 '<br><sub>Один EU-аккаунт за короткий срок совершает N операций Wl через разных агентов</sub>',
            font=dict(size=17)
        ),
        showlegend=True,
        legend=dict(x=1.01, y=1),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        annotations=[
            dict(x=0, y=5.5, text='<b>Украденные аккаунты EU</b>', showarrow=False,
                 font=dict(size=14, color='#EF4444'), bgcolor='rgba(239,68,68,0.1)'),
            dict(x=0, y=-3.2, text='<b>Агенты RET (распределённый вывод)</b>', showarrow=False,
                 font=dict(size=14, color='#F59E0B'), bgcolor='rgba(245,158,11,0.1)')
        ],
        height=650,
        margin=dict(r=220),
        plot_bgcolor='#F8FAFC'
    )
)
fig5.show()

In [93]:
# Временная шкала кражи: демонстрируем скорость вывода
victim_ex = top_victims_sev[0]
df_v = df_sev[df_sev['sender_id'] == victim_ex].sort_values('ts')

# Рассчитываем кумулятивную сумму
df_v['cum_amount'] = df_v['amount'].cumsum()
df_v['minutes_from_start'] = (df_v['ts'] - df_v['ts'].min()).dt.total_seconds() / 60

fig6 = make_subplots(rows=2, cols=1, subplot_titles=[
    f'Суммы отдельных снятий ({victim_ex})',
    'Кумулятивный вывод средств (нарастающим итогом)'
])

fig6.add_trace(go.Bar(
    x=df_v['ts'], y=df_v['amount'],
    name='Сумма снятия',
    marker_color='#EF4444',
    hovertemplate='<b>%{x}</b><br>%{y:,.0f} mM<extra></extra>'
), row=1, col=1)

fig6.add_trace(go.Scatter(
    x=df_v['ts'], y=df_v['cum_amount'],
    mode='lines+markers',
    name='Итого выведено',
    line=dict(color='#F59E0B', width=2),
    hovertemplate='<b>%{x}</b><br>Итого: %{y:,.0f} mM<extra></extra>'
), row=2, col=1)

fig6.update_layout(
    title=f'Детальный анализ кражи телефона: {victim_ex}',
    height=550, showlegend=False
)
fig6.show()

duration = (df_v['ts'].max() - df_v['ts'].min()).total_seconds() / 60
print(f'Жертва: {victim_ex}')
print(f'Всего операций снятия: {len(df_v)}')
print(f'Общая сумма: {df_v["amount"].sum():,.0f} mM')
print(f'Длительность атаки: {duration:.1f} минут')
print(f'Использовано агентов: {df_v["receiver_id"].nunique()}')

Жертва: PN_EU_0_1017
Всего операций снятия: 4
Общая сумма: 96,845 mM
Длительность атаки: 10.9 минут
Использовано агентов: 4


In [94]:
# Pyvis: интерактивная визуализация схемы кражи телефона
net2 = Network(height='500px', width='100%', directed=True, bgcolor='#1E293B', font_color='white')
net2.set_options('''
{
  "physics": {
    "barnesHut": {"gravitationalConstant": -5000, "springLength": 150},
    "stabilization": {"iterations": 150}
  },
  "edges": {"arrows": {"to": {"enabled": true}}}
}
''')

for n in G_scheme2.nodes():
    r = roles2[n]
    color = '#EF4444' if r == 'victim' else '#F59E0B'
    label = n.replace('PN_EU_', 'EU_').replace('PN_Ret', 'RET')
    size = 30 if r == 'victim' else 20
    net2.add_node(n, label=label, color=color, size=size, title=f'{n}\n{role_name.get(r,r)}')

for u, v, data in G_scheme2.edges(data=True):
    w = data.get('weight', 1)
    amt = data.get('amount', 0)
    net2.add_edge(u, v, color='#EF4444', width=w * 0.5 + 0.5,
                  title=f'Снятий: {w}\nИтого: {amt:,.0f} mM')

net2.save_graph('./outputs/scheme2_phone_theft.html')

## 5. Сравнительный анализ двух схем

In [95]:
# Сравнение суммарных характеристик мошеннических схем
schemes = {
    'Ботнет (F_bot)': df_bot,
    'Мул-вывод (F-Mule-With)': df_mule,
    'Кража телефона (F_SevWith)': df_sev
}

comparison = []
for name, d in schemes.items():
    comparison.append({
        'Схема': name,
        'Кол-во транзакций': len(d),
        'Ср. сумма (mM)': d['amount'].mean(),
        'Итого (mM)': d['amount'].sum(),
        'Уникальных отправителей': d['sender_id'].nunique(),
        'Уникальных получателей': d['receiver_id'].nunique()
    })

comp_df = pd.DataFrame(comparison).set_index('Схема')
print(comp_df.to_string())

                            Кол-во транзакций  Ср. сумма (mM)  Итого (mM)  Уникальных отправителей  Уникальных получателей
Схема                                                                                                                     
Ботнет (F_bot)                            721    10689.557268  7707170.79                       39                       4
Мул-вывод (F-Mule-With)                   717    10591.167001  7593866.74                        4                       6
Кража телефона (F_SevWith)                240    13007.547417  3121811.38                       60                       6


In [96]:
# Plotly: сравнительная таблица-диаграмма
metrics = ['Кол-во транзакций', 'Ср. сумма (mM)', 'Уникальных отправителей', 'Уникальных получателей']
colors_comp = ['#EF4444', '#F59E0B', '#8B5CF6']

fig7 = make_subplots(rows=2, cols=2, subplot_titles=metrics)

positions = [(1,1),(1,2),(2,1),(2,2)]
for idx, metric in enumerate(metrics):
    r, c = positions[idx]
    fig7.add_trace(go.Bar(
        x=comp_df.index,
        y=comp_df[metric],
        marker_color=colors_comp,
        showlegend=False,
        hovertemplate='<b>%{x}</b><br>' + metric + ': %{y:,.1f}<extra></extra>'
    ), row=r, col=c)

fig7.update_layout(
    title='Сравнение характеристик мошеннических схем',
    height=550
)
fig7.show()

## 6. Выводы и признаки мошенничества

### Схема №1: Мобильный ботнет + Мул
**Графовый паттерн:** звезда (fan-in) к мулу -> веер (fan-out) к агентам RET  
**Признаки:**
- Один EU-аккаунт (мул) получает переводы от многих разных EU-пользователей (тип Ind/C2C)
- Входящие суммы слегка превышают суммы последующих снятий (мул "хранит" часть)
- Сразу после поступления мул снимает деньги через Wl к агентам RET
- Агенты RET принимают снятия от нескольких мулов
- Временной паттерн: поступление -> снятие с задержкой в часы/дни

### Схема №2: Кража мобильного телефона
**Графовый паттерн:** веер (fan-out) от одного EU к нескольким RET  
**Признаки:**
- Один EU-аккаунт совершает N операций снятия Wl за очень короткий промежуток времени
- Снятия направляются сразу к нескольким разным агентам RET (распределение риска)
- Высокая скорость: серия снятий за минуты, не характерная для обычного поведения
- Суммы снятий близки к балансу аккаунта (жертва не подозревает о краже)
- Нет предшествующей активности входящих переводов - деньги уже были на счёте

Всего транзакций: 54848

Мошеннических: 1678 (3.1%)

Схема 1 (Ботнет+Мул): 4 мула, ~721 ботнет-транзакций + 717 выводов

Схема 2 (Кража телефона): жертвы делают 4-6 снятий за несколько минут